# Plain instructions to code: IIR filtering of IQ data

This notebook implements the natural-language IIR exercise with a canonical IQ layout: `X.shape == (N, 2, L)`, where axis 0 indexes examples, axis 1 contains I/Q, and axis 2 contains time samples.

In [ ]:
from pathlib import Path
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt

In [ ]:
N, L = 5, 1000
SEED = 42
rng = np.random.default_rng(SEED)
time = np.arange(L, dtype=np.float32)
low_frequency = 0.03
high_frequency = 0.35
noise_scale = 0.25
X = np.empty((N, 2, L), dtype=np.float32)
for exampleIndex in range(N):
    phase = 0.4 * exampleIndex
    i_signal = np.sin(2 * np.pi * low_frequency * time + phase)
    q_signal = np.cos(2 * np.pi * low_frequency * time + phase)
    i_interference = 0.5 * np.sin(2 * np.pi * high_frequency * time)
    q_interference = 0.5 * np.cos(2 * np.pi * high_frequency * time)
    X[exampleIndex, 0] = i_signal + i_interference + noise_scale * rng.standard_normal(L)
    X[exampleIndex, 1] = q_signal + q_interference + noise_scale * rng.standard_normal(L)

print(f'Input shape: {X.shape}')
print(f'Input dtype: {X.dtype}')

In [ ]:
ORDER = 2
CUTOFF = 0.1
b, a = signal.butter(ORDER, CUTOFF, btype='lowpass')
X_filtered = signal.lfilter(b, a, X, axis=2).astype(np.float32)

def compute_power(iq_array):
    i_component = iq_array[:, 0, :]
    q_component = iq_array[:, 1, :]
    return float(np.mean(i_component ** 2 + q_component ** 2))

power_before = compute_power(X)
power_after = compute_power(X_filtered)
power_ratio = power_after / power_before
print(f'Power before: {power_before:.6f}')
print(f'Power after: {power_after:.6f}')
print(f'Power ratio: {power_ratio:.6f}')

In [ ]:
example_index = 0
figure, axes = plt.subplots(2, 2, figsize=(12, 8))
figure.suptitle('IIR low-pass filtering of synthetic IQ data')

axes[0, 0].plot(time, X[example_index, 0], alpha=0.65, label='Before')
axes[0, 0].plot(time, X_filtered[example_index, 0], label='After')
axes[0, 0].set_title('I time trace')
axes[0, 0].set_xlabel('Sample')
axes[0, 0].legend()

axes[0, 1].plot(time, X[example_index, 1], alpha=0.65, label='Before')
axes[0, 1].plot(time, X_filtered[example_index, 1], label='After')
axes[0, 1].set_title('Q time trace')
axes[0, 1].set_xlabel('Sample')
axes[0, 1].legend()

axes[1, 0].scatter(X[example_index, 0], X[example_index, 1], s=8, alpha=0.25, label='Before')
axes[1, 0].scatter(X_filtered[example_index, 0], X_filtered[example_index, 1], s=8, alpha=0.5, label='After')
axes[1, 0].set_title('I/Q constellation')
axes[1, 0].set_xlabel('I')
axes[1, 0].set_ylabel('Q')
axes[1, 0].legend()

axes[1, 1].bar(['Before', 'After'], [power_before, power_after])
axes[1, 1].set_title('Mean power comparison')
axes[1, 1].set_ylabel('Mean power')
figure.tight_layout()
plt.show()

In [ ]:
output_path = Path('Novoa') / 'filtered_iq.npz'
output_path.parent.mkdir(parents=True, exist_ok=True)
np.savez(output_path, X_filtered=X_filtered, b=b, a=a, seed=SEED)

assert X_filtered.shape == (N, 2, L)
assert X_filtered.dtype == np.float32
assert not np.allclose(X_filtered, X)
assert power_after < power_before
assert np.load(output_path)['X_filtered'].shape == X_filtered.shape
print(f'Saved: {output_path}')
print('PASS: output shape, dtype, changed signal, reduced power, and NPZ round-trip verified')